# 03 — Expériences : plus longtemps, plus gros

Le notebook 02 a laissé deux constats :

- la loss de validation **baissait encore** au bout de 3 000 pas : le modèle
  n'avait pas fini d'apprendre ;
- il ne reprenait que **17 %** des mots-clés : il lit la consigne, mais
  l'oublie souvent.

On compare ici trois entraînements, comme le script 06 comparait ses cinq
modèles :

| run | modèle | pas |
|---|---|---|
| `A_petit_3k` | celui du notebook 02 (`n_embd` 128, 4 blocs) | 3 000 |
| `B_petit_10k` | le même | 10 000 |
| `C_gros_10k` | plus large et plus profond (`n_embd` 192, 6 blocs) | 10 000 |

`A` contre `B` : l'effet de la **durée**. `B` contre `C` : l'effet de la
**taille**, à durée égale.

Le code du notebook 02 (vocabulaire, `RhymeDataset`, GPT, `write_rhyme`) est
maintenant dans le module `comptines.py` : on l'importe au lieu de le
recopier une troisième fois.

In [ ]:
import copy
import random
import shutil
import time
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, RandomSampler
from torch.utils.tensorboard import SummaryWriter

from comptines import CharTokenizer, GPTLanguageModel, RhymeDataset, keyword_hits, pick_keywords, write_rhyme

In [ ]:
SEED = 1337
torch.manual_seed(SEED)
random.seed(SEED)

try:
    ROOT = Path(__file__).resolve().parent
except NameError:  # exécution dans une cellule Jupyter : `__file__` n'existe pas
    ROOT = Path.cwd()

DATA_DIR = ROOT / "data"
RUNS_DIR = ROOT / "runs" / "03_experiences"
CHECKPOINT_DIR = ROOT / "checkpoints"

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"device : {device}")

## 1. Données

Les mêmes que dans le notebook 02.

In [ ]:
train_rhymes = (DATA_DIR / "comptines_train.txt").read_text(encoding="utf-8").split("\n\n")
val_rhymes = (DATA_DIR / "comptines_val.txt").read_text(encoding="utf-8").split("\n\n")
tokenizer = CharTokenizer(sorted(set("".join(train_rhymes + val_rhymes) + "words: ,\n")))

block_size = 256
batch_size = 16
learning_rate = 1e-3
eval_every = 500
val_batches = 20

train_dataset = RhymeDataset(train_rhymes, tokenizer, block_size)
val_dataset = RhymeDataset(val_rhymes, tokenizer, block_size, fixed_keywords=True)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    sampler=RandomSampler(val_dataset, replacement=True, num_samples=val_batches * batch_size),
)


def make_train_loader(max_steps: int) -> DataLoader:
    """Un `DataLoader` qui tire `max_steps` lots au hasard dans le train."""
    sampler = RandomSampler(train_dataset, replacement=True, num_samples=max_steps * batch_size)
    return DataLoader(train_dataset, batch_size=batch_size, sampler=sampler)


print(f"{len(train_rhymes):,} comptines d'entraînement   {len(val_rhymes):,} de validation   vocab_size : {tokenizer.vocab_size}")

## 2. La boucle, avec le meilleur modèle gardé de côté

La boucle du notebook 02, avec un ajout. En 10 000 pas, chaque comptine
passe une soixantaine de fois : le modèle peut commencer à les apprendre
**par cœur**. On le verrait à la courbe de validation, qui cesserait de
baisser puis remonterait pendant que celle de train continue de descendre.

Pour ne pas perdre le bon modèle dans ce cas, on **garde une copie des poids
à chaque nouveau record de validation**. À la fin, on recharge ce meilleur
modèle plutôt que le dernier. C'est l'*early stopping* : arrêter (ou
revenir) au moment où le modèle généralise le mieux.

In [ ]:
def train_step(model: nn.Module, optimizer: torch.optim.Optimizer, X: torch.Tensor, Y: torch.Tensor) -> float:
    """Un pas d'entraînement sur un lot ; renvoie la loss du lot."""
    model.train()  # dropout actif
    _, loss = model(X.to(device), Y.to(device))  # 1. forward, loss comprise
    optimizer.zero_grad(set_to_none=True)
    loss.backward()  # 2. backward
    optimizer.step()  # 3. step
    return loss.item()


@torch.no_grad()
def test(model: nn.Module, loader: DataLoader) -> float:
    """Mesure la loss moyenne sur les lots du loader, sans apprendre."""
    model.eval()  # dropout coupé
    total = 0.0
    for X, Y in loader:
        _, loss = model(X.to(device), Y.to(device))
        total += loss.item()
    return total / len(loader)


def run_training(model: nn.Module, run: str, max_steps: int) -> dict:
    """Entraîne pendant `max_steps` pas, mesure tous les `eval_every` pas,
    et recharge à la fin les poids du meilleur pas de validation.

    Returns:
        L'historique `(pas, train, val)`, le meilleur pas, sa loss de validation et la durée.
    """
    torch.manual_seed(SEED)
    log_dir = RUNS_DIR / run
    shutil.rmtree(log_dir, ignore_errors=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    train_losses, history = [], []
    best_val, best_step, best_state = float("inf"), 0, None
    start = time.perf_counter()

    with SummaryWriter(log_dir=str(log_dir)) as writer:
        for step, (X, Y) in enumerate(make_train_loader(max_steps), start=1):
            train_losses.append(train_step(model, optimizer, X, Y))

            if step % eval_every == 0:
                train_loss = sum(train_losses) / len(train_losses)
                val_loss = test(model, val_loader)
                writer.add_scalars("loss", {"train": train_loss, "val": val_loss}, step)
                history.append((step, train_loss, val_loss))
                train_losses = []
                if val_loss < best_val:  # nouveau record : on garde une copie des poids
                    best_val, best_step, best_state = val_loss, step, copy.deepcopy(model.state_dict())

    duration = time.perf_counter() - start
    model.load_state_dict(best_state)
    print(f"[{run}] {sum(p.numel() for p in model.parameters()):,} paramètres   {max_steps:,} pas   "
          f"{duration:.0f} s   meilleure val {best_val:.4f} au pas {best_step:,}")
    return {"history": history, "best_step": best_step, "best_val": best_val, "duration": duration}

## 3. Le taux de reprise des mots-clés, sur toute la validation

Au notebook 02, on le mesurait sur 60 comptines, soit environ 120 mots-clés :
une marge d'erreur d'environ ±7 points, trop large pour comparer des modèles
proches. On le mesure ici sur les **299** comptines de validation, avec
leurs mots-clés fixes : les trois modèles reçoivent exactement les mêmes
consignes.

In [ ]:
n_eval = len(val_rhymes)
keyword_sets = [pick_keywords(r, random.Random(i)) for i, r in enumerate(val_rhymes[:n_eval])]
keyword_sets = [k for k in keyword_sets if k]


def evaluate_keywords(model: nn.Module) -> tuple[float, float]:
    """Taux de reprise des mots-clés, et la référence (consigne d'une autre comptine)."""
    torch.manual_seed(SEED)
    generated = [write_rhyme(model, tokenizer, k) for k in keyword_sets]
    hits = sum(keyword_hits(k, g) for k, g in zip(keyword_sets, generated)) / len(generated)
    shifted = generated[1:] + generated[:1]
    baseline = sum(keyword_hits(k, g) for k, g in zip(keyword_sets, shifted)) / len(generated)
    return hits, baseline


print(f"{len(keyword_sets)} consignes, {sum(len(k) for k in keyword_sets)} mots-clés")

## 4. Les trois entraînements

`C_gros_10k` : `n_embd` 192 au lieu de 128, 6 blocs au lieu de 4, 6 têtes
(de 32 dimensions chacune, comme avant). Environ trois fois plus de
paramètres. Le reste ne change pas.

Durée totale sur la RTX 3050 : de l'ordre de 40 minutes.

In [ ]:
small = {"n_embd": 128, "n_head": 4, "n_layer": 4, "dropout": 0.2}
large = {"n_embd": 192, "n_head": 6, "n_layer": 6, "dropout": 0.2}
experiments = {
    "A_petit_3k": (small, 3_000),
    "B_petit_10k": (small, 10_000),
    "C_gros_10k": (large, 10_000),
}

results, models = {}, {}
for run, (arch, max_steps) in experiments.items():
    config = {"vocab_size": tokenizer.vocab_size, "block_size": block_size, **arch}
    model = GPTLanguageModel(**config).to(device)
    results[run] = run_training(model, run, max_steps)
    results[run]["config"] = config
    results[run]["n_params"] = sum(p.numel() for p in model.parameters())
    results[run]["hits"], results[run]["baseline"] = evaluate_keywords(model)
    models[run] = model

## 5. Les courbes

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for run, res in results.items():
    steps, train_curve, val_curve = zip(*res["history"])
    axes[0].plot(steps, train_curve, label=run)
    axes[1].plot(steps, val_curve, label=run)
    axes[1].scatter([res["best_step"]], [res["best_val"]], s=30)
axes[0].set_title("train (dropout actif)")
axes[1].set_title("validation (• : meilleur pas)")
for ax in axes:
    ax.set_xlabel("pas")
    ax.grid(alpha=0.3)
    ax.legend()
axes[0].set_ylabel("loss")
plt.tight_layout()
plt.show()

## 6. Le bilan

In [ ]:
print(f"{'run':14} {'paramètres':>11} {'durée':>7} {'meilleur pas':>13} {'val':>7} {'mots repris':>12} {'référence':>10}")
for run, res in results.items():
    print(f"{run:14} {res['n_params']:>11,} {res['duration']:>6.0f}s {res['best_step']:>13,} "
          f"{res['best_val']:>7.4f} {res['hits']:>12.1%} {res['baseline']:>10.1%}")

## 7. Les comptines du meilleur modèle

Le meilleur modèle est celui qui a la plus basse loss de validation. Mêmes
consignes qu'au notebook 02, pour comparer.

In [ ]:
best_run = min(results, key=lambda r: results[r]["best_val"])
best_model = models[best_run]
print(f"meilleur modèle : {best_run}\n")

torch.manual_seed(SEED)
for words in [["cat", "moon"], ["king", "pie"], ["rain", "garden"], ["dog", "bone", "star"]]:
    print(f"--- {', '.join(words)} ---")
    print(write_rhyme(best_model, tokenizer, words), end="\n\n")

## 8. Sauvegarder

Chaque modèle, dans le même format qu'au notebook 02 : poids, configuration
et vocabulaire.

In [ ]:
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
for run, model in models.items():
    res = results[run]
    torch.save({
        "model_state": model.state_dict(),
        "config": res["config"],
        "chars": tokenizer.chars,
        "val_loss": res["best_val"],
        "keyword_hits": res["hits"],
    }, CHECKPOINT_DIR / f"{run}.pt")
    print(f"→ {(CHECKPOINT_DIR / f'{run}.pt').relative_to(ROOT)}")